# 📗 02. 데이터 EDA — H&M 커머스 탐색적 분석 (강사용 예시)

> 🔧 **강사용 안내**: 이 노트북의 미션 답안엔 **예시 풀이**가 들어 있습니다. **정답이 아니라 "이렇게도 쓸 수 있다"의 한 사례**입니다. 학생 답이 다른 컬럼·다른 그래프를 골랐어도, 요구사항과 체크포인트를 충족했다면 맞습니다.

> 🔧 **배포본과의 차이**: 배포본에는 **실행 코드가 하나도 없습니다** — 2절 가이드까지 전부 "무엇을 만들고 어떤 도구를 쓰는지"만 알려 주는 지침으로 나갑니다. 이 강사용 노트북은 그 지침대로 쓰면 어떤 코드가 되는지 보여주는 참고본이며, 위에서 아래로 통실행됩니다.

이 노트북은 **01_데이터이해_전처리**가 만든 정제본으로 **탐색적 데이터 분석(EDA)** 을 합니다. day07(EDA·시각화) 전체와 day08(기술통계)을 이 파트에서 다룹니다 — **추론통계(가설검정·회귀)는 03_데이터_통계**에서 이어집니다.

**이 노트북 읽는 법**
- 각 절 시작에 **"이 절에서 할 일"** 한 줄이 있습니다.
- 막히면 **"막히면 볼 곳"** 이 가리키는 가이드 절로 가서 예시 코드를 확인하세요.
- 각 절 끝에 **"여기까지 되면 통과"** 로 스스로 확인하세요(자가채점 없음).
- **3절(필수 미션)은 반드시**, **4절(선택 카탈로그)은 원하는 것만** — 이 둘은 성격이 다르니 헷갈리지 마세요.

**목차**
1. 정제본 불러오기
2. 가이드 — 집계·시각화·기술통계 도구
3. 필수 EDA 미션 (반드시)
4. 선택 EDA 미션 카탈로그 (골라서)
5. 관찰 정리 (서술)

## 1. 정제본 불러오기
**이 절에서 할 일**: 01 이 저장한 정제본(`output/hm_clean.csv`)을 불러와, 이후 모든 절에서 이어 쓸 `df` 를 준비합니다.

> ⚠️ **01 을 먼저 끝내는 것이 원칙입니다.** 정제본이 없으면 아래 코드가 **표준 규칙으로 그 자리에서 재현**하지만, 이 재현 경로는 **여러분이 01 에서 고른 처리 규칙(다른 연령 기준, 다른 결측 처리 등)을 반영하지 못합니다** — 항상 01 을 먼저 실행하고 오세요.

In [ ]:
# [제공 코드] 이 노트북에서 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')
import os
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=KOREAN_FONT, rc={'axes.unicode_minus': False})

In [ ]:
# [제공 코드] 정제본 로드 — 있으면 그대로 쓰고, 없으면 표준 규칙으로 그 자리에서 재현합니다.
clean_path = 'output/hm_clean.csv'
# 이 노트북의 예시가 쓰는 컬럼 — 하나라도 없으면 예시를 돌릴 수 없어 표준 규칙으로 재현합니다.
NEEDED = ['t_dat', 'age', 'price', 'sales_channel_id', 'club_member_status']

df = None
if os.path.exists(clean_path):
    loaded = pd.read_csv(clean_path)
    lack = [c for c in NEEDED if c not in loaded.columns]
    if lack:
        print(f"정제본에 이 노트북의 예시가 쓰는 컬럼이 없습니다: {', '.join(lack)}")
        print('예시를 돌리기 위해 표준 규칙으로 재현합니다.')
        print('여러분의 정제본을 그대로 쓰려면 01 에서 이 컬럼들을 남긴 채 다시 저장하세요.')
    else:
        df = loaded
        print('01 의 정제본을 불러왔습니다:', df.shape)

        # 01 에서 남겨 두기로 한 값이 있으면 이 노트북의 예시를 위해서만 걸러내고,
        # 무엇을 왜 걸렀는지 알려 줍니다(여러분의 선택을 부정하는 것이 아닙니다).
        fixed = []
        bad_age = (df['age'] < 10) | (df['age'] > 99)
        if bad_age.any():
            fixed.append(f'비현실 나이 {bad_age.sum()} 행')
            df = df[~bad_age]
        bad_price = df['price'] <= 0
        if bad_price.any():
            fixed.append(f'0 이하 가격 {bad_price.sum()} 행')
            df = df[~bad_price]
        bad_ch = ~df['sales_channel_id'].isin([1, 2])
        if bad_ch.any():
            fixed.append(f'정의에 없는 채널 코드 {bad_ch.sum()} 행')
            df = df[~bad_ch]
        if df['club_member_status'].isna().any():
            fixed.append('멤버십 결측 -> Unknown')
            df['club_member_status'] = df['club_member_status'].fillna('Unknown')
        if fixed:
            print('아래 항목이 남아 있어, 이 노트북의 예시를 돌리기 위해 여기서만 걸러 썼습니다.')
            print('(01 에 기록한 여러분의 선택은 바뀌지 않습니다 — 이 df 에만 적용되는 임시 조치입니다.)')
            for item in fixed:
                print('  -', item)
            print('예시용:', df.shape)
            print('  - 이 규칙을 의도해서 고른 것이라면 그대로 진행하세요.')
            print('    (그 값들을 살려 분석하려면 아래 레시피의 필터를 여러분 규칙에 맞게 고쳐 쓰면 됩니다.)')
            print('  - 01 을 아직 안 끝냈거나 실수로 빠뜨린 것이라면, 01 을 완료한 뒤 다시 저장하세요.')

if df is None:
    print('표준 규칙으로 재현합니다 — 원칙은 01 을 먼저 끝내는 것입니다.')
    tr = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/transactions_hm.csv')
    cu = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/customer_hm.csv')
    ar = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/articles_hm.csv')
    df = tr.merge(cu, on='customer_id', how='inner').merge(ar, on='article_id', how='inner')
    df = df.drop_duplicates()
    df = df[(df['age'] >= 10) & (df['age'] <= 99)]      # 비현실 연령 제거
    df = df[df['price'] > 0]                             # 0원 거래 제외
    df = df[df['sales_channel_id'].isin([1, 2])]         # 채널 코드 정상값만
    df['club_member_status'] = df['club_member_status'].fillna('Unknown')
    print('표준 규칙 재현 후:', df.shape)

df['t_dat'] = pd.to_datetime(df['t_dat'])   # CSV 왕복하며 날짜형이 문자열로 풀렸으면 다시 변환
display(df.head(3))

> ⚠️ **발제문 설명과 데이터가 다르면, 옳은 것은 데이터입니다.** 발제문 참고사항에는 `price` 가 **"SEK(스웨덴 크로나)"** 이고 기간이 **"수년간"** 이라고 적혀 있습니다. 직접 확인해 보세요 — `df['price'].min()`·`max()` 와 `df['t_dat'].min()`·`max()` 를 찍어 보면, `price` 는 0 과 1 사이의 **정규화된 상대값**이고 기간은 **1년**입니다. 그래서 이 프로젝트는 price 를 금액이 아니라 **상대값**으로만 다루고("몇 크로나"가 아니라 "어느 쪽이 몇 배"), **"연도별 추세"는 이 데이터로 말하지 않습니다.** 남이 써 준 설명을 데이터로 검증하는 것도 분석가의 일입니다.

**파생변수 만들기 — 이후 절에서 쓸 `month`·`weekday`·`age_group`**
정제본에 이미 있어도 여기서 다시 확인해 둡니다. 이후 EDA·기술통계에서 계속 쓸 파생변수 세 개입니다(day07 파생변수 레시피 — 날짜 분해·구간화). 이 노트북의 나머지 절은 전부 이 파생변수가 있다고 가정합니다.

In [ ]:
# [레시피] 파생변수 — 날짜 분해(월·요일) + 구간화(10살 단위 연령대)
df['month'] = df['t_dat'].dt.month
df['weekday'] = df['t_dat'].dt.day_name()
df['age_group'] = (df['age'] // 10 * 10).astype(int)
df_s = df.sample(n=3000, random_state=42)   # 무거운 그림·집계에 재사용할 표본
print('파생변수 확인 후:', df.shape)

**짚고 넘어가기 — merge 후 행이 늘지 않았는지 검증**
01 에서 거래·고객·상품을 merge 할 때 다대다(many-to-many) 매칭이 있으면 행이 **예상보다 훨씬 많이** 늘어납니다(뻥튀기). 지금 정제본을 받았어도, 습관적으로 **거래를 유일하게 식별하는 조합**(고객·상품·날짜)에 중복이 없는지 한 번 더 확인하세요.

In [ ]:
# [레시피] 거래 식별 조합(customer_id, article_id, t_dat)의 중복 개수 확인
dup_combo = df.duplicated(subset=['customer_id', 'article_id', 't_dat']).sum()
print(f'거래 식별 조합 중복: {dup_combo}건 / 전체 {len(df)}행')

> ⚠️ 0 이 아니어도 바로 오류는 아닙니다 — 같은 고객이 같은 날 같은 상품을 **여러 번** 사면 똑같은 조합이 나올 수 있습니다(수량 컬럼이 없을 때). 이 정제본은 그 비율이 전체 대비 매우 작습니다 — **비율이 크다면** merge 방식(다대다 매칭 여부)을 의심하세요.

> ✅ **여기까지 되면 통과**: `df.shape` 가 출력되고 위 `head()` 표가 정상적으로 보이면 됩니다(정확한 행수는 01 에서 고른 규칙에 따라 달라질 수 있습니다).

## 2. 가이드 — 막힐 때 여기를 보세요
**이 절에서 할 일**: 없습니다 — 이 절은 **참고용 레시피 모음**입니다. 순서대로 읽지 말고, 3·4절을 하다 막히면 필요한 부분만 찾아 복사해 쓰세요. 아래 예시는 모두 위 1절에서 로드한 **정제본 `df`** 로 실제 실행됩니다.

### 2-1. 집계 도구 선택 기준 — `groupby` vs `pivot_table` vs `crosstab`
셋 다 "그룹으로 묶어 본다"는 같은 아이디어지만, **무엇을 보고 싶은지**에 따라 고릅니다.

| 하고 싶은 것 | 도구 |
|---|---|
| 그룹별로 **여러 통계**(평균·합계·개수 등)를 한 표로 | `groupby` + `agg` |
| **두 범주형** 변수를 행·열로 펼쳐 **수치형 값**(주로 평균·합계)을 비교 | `pivot_table` |
| **두 범주형** 변수의 **빈도(건수)** 를 표로, 필요하면 비율까지 | `crosstab` |

In [ ]:
# [레시피] groupby + agg — 여러 통계를 한 표로
agg_tbl = (df.groupby('product_group_name', observed=True)
             .agg(건수=('price', 'count'), 평균가=('price', 'mean'), 총매출=('price', 'sum'))
             .sort_values('총매출', ascending=False))
display(agg_tbl.head(8))

# [레시피] pivot_table — 두 범주형 변수 x 수치형 값
piv = pd.pivot_table(df, index='age_group', columns='sales_channel_id', values='price', aggfunc='mean')
display(piv.round(4))

# [레시피] crosstab — 두 범주형 변수의 빈도 (normalize='index' 로 비율)
ct = pd.crosstab(df['club_member_status'], df['sales_channel_id'], normalize='index')
display(ct.round(3))

# [레시피] nlargest — 상위 N (sort_values().head(n) 과 결과가 같은, 더 짧은 표현)
top10 = df.groupby('prod_name')['price'].sum().nlargest(10)
display(top10)

# [레시피] 비율 계산 — 전체 대비 비중
share_channel = df['sales_channel_id'].value_counts(normalize=True).round(3)
display(share_channel)

### 2-2. 커스텀 집계 — `lambda`·사용자 함수
평균·합계·개수로 안 되는 질문("범위가 얼마나 되나", "상위 10% 비율은?", "가장 흔한 가격대는?")은 `agg` 에 **함수를 직접 넣어** 구합니다.

In [ ]:
# [레시피] 연령대별 — 범위 · 상위 10% 비율 · 최빈 가격대 (직접 만든 함수)
custom_agg = df.groupby('age_group', observed=True)['price'].agg(
    범위=lambda s: s.max() - s.min(),
    고가비율=lambda s: (s > s.quantile(0.9)).mean(),
    최빈가격대=lambda s: s.round(2).mode().iloc[0],
)
display(custom_agg.round(3))

### 2-3. 시각화 갤러리 — 변수 개수·유형으로 고르기
그래프를 고를 때 헤매지 않는 방법은 **"몇 개의 변수를, 어떤 유형으로 볼 것인가"** 를 먼저 정하는 것입니다. 무거운 그림은 전체 대신 **3,000행 표본**(1절에서 만든 `df_s`)으로 그립니다 — 경향은 같고 훨씬 빠릅니다.

| 변수 개수 | 유형 | 그래프 |
|---|---|---|
| **단변량** | 수치 | `histplot`·`boxplot`(+ 로그축) |
| **단변량** | 범주 | `countplot` |
| **이변량** | 수치 × 수치 | `scatterplot`·`regplot` |
| **이변량** | 범주 × 수치 | `barplot`·`boxplot`(+ `hue`) |
| **이변량** | 범주 × 범주 | `crosstab` + `heatmap` |
| **다변량** | 셋 이상 | `hue`·`subplots`(소형 다중 그림)·상관 `heatmap` |
| **시간** | 추세 | `lineplot`·롤링 평균 |

**단변량 — 수치: `histplot`·`boxplot`**
**언제**: 한 수치 변수가 **어떻게 퍼져 있는지**(치우침·봉우리 개수) 볼 때 → `histplot`. **중앙값·사분위·이상치**를 한눈에 볼 때 → `boxplot`.

In [ ]:
# [레시피] 분포 — 오른쪽으로 치우친 값은 로그축(log_scale=True)이 더 잘 보입니다.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df_s['price'], bins=30, ax=axes[0])
axes[0].set_title('price 분포 (원래 축)')
sns.histplot(df_s['price'], bins=30, log_scale=True, ax=axes[1])
axes[1].set_title('price 분포 (로그축)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(data=df_s, x='age_group', y='price')
plt.title('연령대별 price 상자그림')
plt.show()

**단변량 — 범주: `countplot`**
**언제**: 범주별 **건수(빈도)** 만 볼 때 — 집계 없이 원본을 바로 넣습니다.

In [ ]:
# [레시피] 채널별 거래 건수
plt.figure(figsize=(5, 4))
sns.countplot(data=df_s, x='sales_channel_id')
plt.title('채널별 거래 건수')
plt.show()

**이변량 — 수치 x 수치: `scatterplot`·`regplot`**
**언제**: 두 수치 변수의 **관계**를 점으로 볼 때 → `scatterplot`(`hue` 로 세 번째 변수도). **추세선까지** 보고 싶을 때 → `regplot`.

In [ ]:
# [레시피] 나이 vs price 산점도 + 회귀선
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df_s, x='age', y='price', hue='sales_channel_id', alpha=0.4, ax=axes[0])
axes[0].set_title('나이 vs price (채널별 색 구분)')
sns.regplot(data=df_s, x='age', y='price', scatter_kws={'alpha': 0.3}, ax=axes[1])
axes[1].set_title('나이 vs price 회귀선')
plt.tight_layout()
plt.show()

**이변량 — 범주 x 수치: `barplot`·`boxplot`(+ `hue`)**
**언제**: 범주별 **평균 같은 대표값**을 비교할 때 → `barplot`(신뢰구간까지 자동). 범주별 **분포 전체**를 비교하고, 거기에 **세 번째 범주까지** 겹쳐 보고 싶을 때 → `boxplot(hue=...)`.

In [ ]:
# [레시피] 제품군별 평균가
plt.figure(figsize=(9, 4))
order = df.groupby('product_group_name', observed=True)['price'].mean().sort_values(ascending=False).index
sns.barplot(data=df_s, x='product_group_name', y='price', order=order)
plt.xticks(rotation=45, ha='right')
plt.title('제품군별 평균가')
plt.tight_layout()
plt.show()

# [레시피] 채널별 price 분포, 멤버십 상태로 한 번 더 나눠 보기(hue)
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_s, x='sales_channel_id', y='price', hue='club_member_status')
plt.title('채널 x 멤버십별 price 분포')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

**이변량 — 범주 x 범주: `crosstab` + `heatmap`**
**언제**: 두 범주형 변수의 조합별 **건수**를 색으로 한눈에 볼 때.

In [ ]:
# [레시피] 연령대 x 채널 거래 건수 히트맵
ct_age_ch = pd.crosstab(df['age_group'], df['sales_channel_id'])
plt.figure(figsize=(5, 4))
sns.heatmap(ct_age_ch, annot=True, fmt='d', cmap='Blues')
plt.title('연령대 x 채널 거래 건수')
plt.show()

**다변량 — `hue`·`subplots`·상관 `heatmap`**
**언제**: "전체 그림이 그룹마다 다르게 보이는가?" 를 확인할 때. 그룹이 **2~3개**면 `hue` 로 한 그림에 겹쳐 보고, **여러 개**면 `subplots` 로 나란히(소형 다중 그림). **여러 수치 변수 사이 관계를 한 번에** 보려면 상관 `heatmap`.

In [ ]:
# [레시피] hue — 한 그림에 겹쳐서
plt.figure(figsize=(8, 4))
sns.histplot(data=df_s, x='price', hue='sales_channel_id', bins=30, log_scale=True, element='step')
plt.title('채널별 price 분포 (겹쳐 보기)')
plt.show()

# [레시피] subplots — 그룹마다 따로따로 (소형 다중 그림)
groups = df_s['index_group_name'].unique()
fig, axes = plt.subplots(1, len(groups), figsize=(3.2 * len(groups), 3.2), sharey=True)
for ax, g in zip(axes, groups):
    sns.histplot(df_s.loc[df_s['index_group_name'] == g, 'price'], bins=20, ax=ax, log_scale=True)
    ax.set_title(g, fontsize=9)
plt.tight_layout()
plt.show()

# [레시피] 상관 히트맵 — 수치 변수 여러 개를 한 번에 (2-4 기술통계의 '상관'에서 다시 씁니다)
plt.figure(figsize=(5, 4))
sns.heatmap(df_s[['price', 'age', 'month']].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('상관행렬')
plt.show()

**시간 — `lineplot`·롤링 평균**
**언제**: 시간(월·일 등)에 따른 **변화 추이**를 볼 때. `lineplot` 은 x 값이 같은 여러 점을 자동으로 평균 내 그려줍니다. 하루하루가 들쭉날쭉하면 **롤링(이동) 평균**으로 큰 흐름만 봅니다.

In [ ]:
# [레시피] 월별 평균가 추세
monthly = df.groupby('month')['price'].mean().reset_index()
plt.figure(figsize=(8, 4))
sns.lineplot(data=monthly, x='month', y='price', marker='o')
plt.title('월별 평균가 추세')
plt.xticks(range(1, 13))
plt.show()

# [레시피] 일별 평균가 + 7일 롤링 평균 (하루 단위는 들쭉날쭉 -> 롤링으로 큰 흐름만)
daily = df.groupby(df['t_dat'].dt.date)['price'].mean().reset_index(name='avg_price')
daily['t_dat'] = pd.to_datetime(daily['t_dat'])
daily = daily.sort_values('t_dat')
daily['rolling7'] = daily['avg_price'].rolling(7).mean()
plt.figure(figsize=(9, 4))
plt.plot(daily['t_dat'], daily['avg_price'], alpha=0.3, label='일별 평균')
plt.plot(daily['t_dat'], daily['rolling7'], color='crimson', label='7일 롤링 평균')
plt.title('일별 평균가와 7일 롤링 평균')
plt.legend()
plt.show()

### 2-4. 기술통계 도구 (day08)
**추론통계(검정)는 03 담당**입니다 — 여기서는 데이터를 **요약해서 설명**하는 도구만 다룹니다.

**대표값 — 평균·중앙값·최빈값·절사평균·가중평균**
**평균의 함정**: 평균은 **극단값에 약합니다**. `price` 처럼 소수의 아주 비싼 거래가 있으면 평균이 중앙값보다 위로 끌려 올라갑니다 — 이럴 땐 **중앙값**이나, 양 끝 극단치를 잘라내는 **절사평균**(`stats.trim_mean`)이 더 안정적입니다.

In [ ]:
# [레시피] 대표값 네 가지 + 가중평균
mean_p, median_p, mode_p = df['price'].mean(), df['price'].median(), df['price'].mode().iloc[0]
trim_p = stats.trim_mean(df['price'], 0.1)   # 위·아래 10%씩 잘라낸 평균
weighted_p = np.average(df['price'], weights=df['age'])   # 나이로 가중
print(f'평균 {mean_p:.4f} / 중앙값 {median_p:.4f} / 최빈값 {mode_p:.4f} / '
      f'절사평균(10%) {trim_p:.4f} / 가중평균(age) {weighted_p:.4f}')
print('평균이 중앙값보다 높다 -> 소수의 고가 거래가 평균을 끌어올리는 오른쪽 꼬리 분포')

**산포 — 범위·IQR·분산/표준편차·변동계수(CV)**
표준편차는 **단위가 있어서** 서로 다른 변수끼리 "어느 쪽이 더 퍼져 있나" 비교하기 어렵습니다. 이럴 때 **변동계수 CV = 표준편차 / 평균**(단위 없음)으로 비교합니다.

In [ ]:
# [레시피] 범위 · IQR · 표준편차(ddof=1, 표본 기준) · CV
rng = df['price'].max() - df['price'].min()
q1, q3 = df['price'].quantile([0.25, 0.75])
iqr = q3 - q1
std_price = df['price'].std(ddof=1)
cv_price = std_price / df['price'].mean()
cv_age = df['age'].std(ddof=1) / df['age'].mean()
print(f'price - 범위 {rng:.4f} / IQR {iqr:.4f} / 표준편차 {std_price:.4f} / '
      f'CV {cv_price:.3f}   (age CV {cv_age:.3f})')
print('price 의 CV 가 age 의 CV 보다 크다 -> price 가 상대적으로 더 퍼져 있다(단위가 달라 std 만으론 비교 불가)')

**분포 형태 — 왜도·첨도, 로그 변환**
**왜도(skewness)**: 0 이면 좌우 대칭, 양수면 오른쪽 꼬리(고가 이상치)가 깁니다. **첨도(kurtosis)**: 0(정규분포 기준)보다 크면 꼬리가 두꺼워 **극단값이 잦다**는 뜻입니다 — "꼬리 리스크". 오른쪽으로 심하게 치우친 변수는 **로그 변환**으로 대칭에 가깝게 만들 수 있습니다.

In [ ]:
# [레시피] 왜도·첨도, 로그 변환 전후 비교
skew_raw, kurt_raw = stats.skew(df['price']), stats.kurtosis(df['price'])
log_price = np.log(df['price'])   # price > 0 이 보장되어 있어 log1p 없이 바로 log
skew_log, kurt_log = stats.skew(log_price), stats.kurtosis(log_price)
print(f'원본 — 왜도 {skew_raw:.2f}, 첨도 {kurt_raw:.2f}   /   '
      f'로그변환 후 — 왜도 {skew_log:.2f}, 첨도 {kurt_log:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df_s['price'], bins=30, ax=axes[0])
axes[0].set_title(f'원본 (왜도 {skew_raw:.2f})')
sns.histplot(np.log(df_s['price']), bins=30, ax=axes[1])
axes[1].set_title(f'로그 변환 후 (왜도 {skew_log:.2f})')
plt.tight_layout()
plt.show()

**상관 — 피어슨 vs 스피어만, 상관 ≠ 인과**
**피어슨**: 두 변수의 **선형** 관계 강도(직선에 얼마나 가까운지). **스피어만**: 값 대신 **순위**로 계산해 **단조(monotonic) 관계**(반드시 직선이 아니어도 한쪽이 늘 때 다른 쪽도 느는지)를 봅니다 — 이상치에 덜 민감합니다. **둘의 차이가 크면** 관계가 있어도 **비선형**이라는 신호입니다.

> ⚠️ **상관은 인과가 아닙니다.** 두 변수가 함께 움직인다고 한쪽이 다른 쪽의 **원인**이라는 뜻은 아닙니다 — 제3의 변수(공통 원인)나 우연일 수 있습니다.

In [ ]:
# [레시피] 피어슨 vs 스피어만
pearson_r = df[['price', 'age']].corr(method='pearson').iloc[0, 1]
spearman_r = df[['price', 'age']].corr(method='spearman').iloc[0, 1]
print(f'Pearson r = {pearson_r:.3f} / Spearman rho = {spearman_r:.3f}')
print('두 값이 비슷하다 -> 나이-price 관계에 뚜렷한 비선형 패턴은 없다(관계 자체는 약함)')
print('상관행렬 히트맵은 위 2-3 다변량 절의 heatmap 레시피를 그대로 쓰면 됩니다.')

### 2-5. 자주 나는 에러와 해결
**`SettingWithCopyWarning`**: 필터링한 결과에 값을 대입하면 나옵니다 — 계속 수정할 거라면 `.copy()` 를 붙이세요.

In [ ]:
# [해결] 필터링 후 수정할 계획이면 .copy() 를 붙입니다.
channel1 = df[df['sales_channel_id'] == 1].copy()
channel1['price_krw_est'] = channel1['price'] * 1000
print('안전하게 새 컬럼 추가:', 'price_krw_est' in channel1.columns)

**그래프 한글이 깨져 보임(네모 □)**: 1절 맨 위 준비 셀에서 이미 OS 별 폰트를 설정했습니다 — 그래도 깨지면 아래로 확인하세요.

In [ ]:
# [해결] 현재 설정된 폰트 확인
print('현재 폰트:', plt.rcParams['font.family'])

**`groupby` 뒤에 원하는 컬럼이 안 보임(인덱스로 숨어 있음)**: 그룹 키가 **인덱스**로 들어갑니다. 다시 **일반 컬럼**으로 쓰려면(다른 표와 merge, 그래프의 x축 등) `reset_index()` 를 붙이세요.

In [ ]:
# [해결] reset_index() 로 그룹 키를 다시 일반 컬럼으로
g = df.groupby('age_group')['price'].mean()
print('reset_index 전:', type(g.index).__name__)
g2 = g.reset_index()
print('reset_index 후 컬럼:', g2.columns.tolist())

## 3. 필수 EDA 미션
> ⚠️ **이 절은 필수입니다.** 발제문의 필수 분석 목표 중 이 파트가 맡은 두 가지입니다. **자가채점은 없습니다** — 체크포인트로 스스로 확인하세요.

### 필수 미션 1 — 기술통계 제시
정제본의 **컬럼 타입·결측**과, 주요 수치 컬럼(`price`·`age` 등)의 **대표값**(평균·중앙값 등)과 **산포**(표준편차 등)를 표로 제시하세요.

> 🔧 **막히면 볼 곳**: 2-4 기술통계 도구(대표값·산포).

> ✅ **여기까지 되면 통과**: 수치형은 min/median/mean/max, 범주형은 value_counts 상위 몇 개가 나오면 됩니다 — 형식은 자유입니다.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 수치형 요약
num_cols = ['price', 'age']
summary = pd.DataFrame({
    'dtype': df[num_cols].dtypes.astype(str),
    'min': df[num_cols].min(),
    'median': df[num_cols].median(),
    'mean': df[num_cols].mean().round(4),
    'max': df[num_cols].max(),
    '결측': df[num_cols].isna().sum(),
})
display(summary)

# 범주형 요약 — 상위 범주만
for col in ['product_group_name', 'club_member_status', 'sales_channel_id']:
    print(f'[{col}] 상위 범주')
    display(df[col].value_counts().head(5))

> 🔧 **강사 Note**: 숫자보다 '읽고 무엇을 알았는가'가 중요합니다. 예: age 는 16~95세로 넓게 퍼져 있고 중앙값이 32세로 평균(36.3세)보다 낮아 오른쪽으로 약간 치우쳐 있습니다.

### 필수 미션 2 — 기준 컬럼 비교분석 + 시각화
**기준 컬럼을 하나 이상 골라**(연령대·채널·상품군 중, 또는 다른 컬럼도 좋습니다) **집계함수로 비교분석**하고, **최소 3개 이상의 시각화**로 보여주세요(2-3 절의 변수 유형 표를 참고해 단변량·이변량을 섞어도 좋습니다). 그림마다 **무엇을 말하는지 한 줄**을 답니다.

> 🔧 **막히면 볼 곳**: 2-1 집계 도구 선택 기준, 2-3 시각화 갤러리.

> ✅ **여기까지 되면 통과**: 고른 기준 컬럼으로 최소 하나의 groupby/pivot_table 집계 표 + 3개 이상 그림 + 각 그림 아래 한 줄 해석이 있으면 됩니다.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 기준 컬럼: age_group (연령대) — 다른 컬럼을 골랐어도 같은 흐름으로 하면 됩니다.
agg_age = df.groupby('age_group', observed=True).agg(
    건수=('price', 'count'), 평균가=('price', 'mean'), 총매출=('price', 'sum'))
display(agg_age)

# 그림 1 — 연령대별 평균가 (이변량: 범주 x 수치)
plt.figure(figsize=(8, 4))
sns.barplot(data=df, x='age_group', y='price')
plt.title('연령대별 평균가')
plt.show()

# 그림 2 — 연령대별 price 분포(상자그림, 단변량 수치를 그룹별로)
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_s, x='age_group', y='price')
plt.title('연령대별 price 분포')
plt.show()

# 그림 3 — 연령대 x 채널 교차 (이변량: 범주 x 범주)
ct_age_ch = pd.crosstab(df['age_group'], df['sales_channel_id'])
plt.figure(figsize=(5, 4))
sns.heatmap(ct_age_ch, annot=True, fmt='d', cmap='Blues')
plt.title('연령대 x 채널 거래 건수')
plt.show()

> 🔧 **강사 Note**: 해석 예시 — 1) 20대가 건수·매출 모두 가장 크다(총매출 1위). 2) 연령대가 높아질수록 평균가가 소폭 상승한다(30대 이후 완만한 우상향). 3) 모든 연령대에서 채널 2(온라인)가 채널 1(오프라인)보다 건수가 많다 — 특히 20대에서 그 격차가 가장 크다.

## 4. 선택 EDA 미션 카탈로그
> 🔧 **이 절은 선택입니다.** 원하는 것만 골라서 하세요 — 전부 할 필요 없습니다. 각 미션은 **발제문의 심화 분석 목표**를 문제로 만든 것이고, 2절의 도구 지도를 응용하면 대부분 시작할 수 있습니다.

| 난이도 | 선택 미션 | 쓰는 도구 | 예상 시간 |
|---|---|---|---|
| ★ | 1. 산점도 등 다양한 형식으로 시각화 | `scatterplot`·`pairplot` | 15분 |
| ★ | 2. 고객 속성 × 채널 교차분석 | `crosstab`·`heatmap` | 15분 |
| ★★ | 3. 로그 변환·정규화(z-score·Min-Max)로 분포 비교 | `np.log`·표준화 계산 | 20분 |
| ★★ | 4. 이상치 고객·거래 탐색 전략 | IQR 규칙·`boxplot` | 20분 |
| ★★ | 5. 월별·분기별 평균 매출 — 추세와 시즌성 | `dt.quarter`·`lineplot` | 20분 |
| ★★ | 6. 채널 비중 해석 보정 — 비율(%)과 ARPU | `nunique`·비율 계산 | 20분 |
| ★★★ | 7. 고객별 총 구매액 상위 20% vs 하위 20% | `qcut`·`groupby` | 25분 |
| ★★★ | 8. 월별 매출의 3개월 롤링 평균 | `rolling().mean()` | 20분 |

> ⚠️ **이 데이터의 기간은 1년(2019-01-01 ~ 2019-12-31)입니다.** 그래서 5·8번은 **"연도별 추세"가 아니라 "1년 안의 월별 흐름"** 까지만 말할 수 있습니다. 발제문에는 "수년간"이라고 적혀 있지만, 여러분의 데이터로 직접 확인한 사실이 우선입니다.

### 선택 미션 1 (★) — 산점도 등 다양한 형식으로 시각화
> 발제문 심화 분석 목표 1

**비즈니스 질문**: "많이 사는 고객"과 "비싸게 사는 고객"은 같은 사람인가?

- **무엇을 만드나**: **고객 한 명을 점 하나로** 만든 표(구매건수·총구매액·평균구매액)를 준비해 산점도를 그리고, 세 변수를 한 번에 보는 격자 그림(`pairplot`)까지 그립니다. 거래 한 건이 아니라 **고객 한 명**이 관측 단위가 되는 것이 핵심입니다.
- **어떤 도구를 쓰나**:
  - 고객 단위 표: `df.groupby('customer_id').agg(…)` 로 건수·합계를 구하고, 평균구매액은 두 컬럼을 나눠 만듭니다.
  - 산점도: `sns.scatterplot(data=…, x=…, y=…, alpha=…)` — 점이 많으면 `<표>.sample(n=…, random_state=42)` 로 표본을 씁니다.
  - 여러 수치 변수를 한 번에: `sns.pairplot(<수치 컬럼만 남긴 표>, height=…)` — 대각선은 각 변수의 분포, 나머지 칸은 두 변수의 산점도입니다.
  - 관계의 세기를 숫자로도 보고 싶으면 2-4 절의 상관 도구.
- **이렇게 나오면 맞다**: 산점도에서 구매건수가 늘면 총구매액도 대체로 늘지만(우상향), **평균구매액과 구매건수 사이에는 그런 경향이 거의 안 보입니다** — 이 두 그림의 차이를 한 줄로 설명할 수 있으면 성공입니다.
- **함정**: 고객 대부분이 구매 1~2건이라 점이 왼쪽에 뭉칩니다 — `alpha` 로 투명하게 하거나 표본을 쓰세요. `pairplot` 은 그림 단위 함수라 `figsize` 대신 `height` 를 씁니다(수만 행을 그대로 넣으면 매우 느립니다).

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 관측 단위를 '거래'에서 '고객'으로 바꾼다 — 고객 한 명이 점 하나
cust = df.groupby('customer_id').agg(
    구매건수=('price', 'count'), 총구매액=('price', 'sum'))
cust['평균구매액'] = cust['총구매액'] / cust['구매건수']
print('고객 수:', len(cust))
display(cust.describe().round(4))

cust_s = cust.sample(n=800, random_state=42)   # 점이 겹치지 않게 표본으로
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=cust_s, x='구매건수', y='총구매액', alpha=0.5, ax=axes[0])
axes[0].set_title('구매건수 vs 총구매액')
sns.scatterplot(data=cust_s, x='구매건수', y='평균구매액', alpha=0.5, ax=axes[1])
axes[1].set_title('구매건수 vs 평균구매액')
plt.tight_layout()
plt.show()

display(cust[['구매건수', '총구매액', '평균구매액']].corr().round(3))

# 세 변수를 한 번에 — pairplot (그림 단위 함수: figsize 대신 height)
sns.pairplot(cust_s, height=2.0, plot_kws={'alpha': 0.4})
plt.show()

> 🔧 **강사 Note**: 실제 값 — 구매건수와 총구매액의 상관은 0.602, 구매건수와 평균구매액은 0.018. "많이 사는 고객"은 총액이 크지만 **거래당 단가는 다르지 않다**는 뜻입니다. 여기서 "고객을 총액으로 나눌 것인가, 단가로 나눌 것인가"라는 세그먼트 질문이 나옵니다(선택 미션 7로 이어집니다).

### 선택 미션 2 (★) — 고객 속성 × 채널 교차분석
> 발제문 심화 분석 목표 7

**비즈니스 질문**: 멤버십 상태나 뉴스 구독 습관에 따라 **주로 쓰는 채널**이 다른가?

- **무엇을 만드나**: 고객 속성(`club_member_status`·`fashion_news_frequency`·`age_group` 중 둘 이상)과 채널의 교차표를 만들고, **행 기준 비율**로 바꿔 히트맵으로 보여줍니다. 비율표와 함께 **건수표도** 제시해 표본이 작은 범주를 드러냅니다.
- **어떤 도구를 쓰나**:
  - 교차표: `pd.crosstab(<속성 시리즈>, <채널 시리즈>)` — 비율은 `normalize='index'`.
  - 히트맵: `sns.heatmap(<비율표>, annot=True, fmt='.2f', cmap='Blues')`.
  - 표본 크기 확인: 비율표와 나란히 `normalize` 없는 건수표를 함께 출력합니다.
- **이렇게 나오면 맞다**: 비율표와 건수표를 나란히 놓았을 때, **비중이 극단적으로 보이는 범주가 실은 건수가 아주 적다**는 것을 스스로 지적할 수 있으면 성공입니다. 어떤 속성에서는 범주 간 비중 차이가 거의 없을 수도 있는데, 그 "차이가 없다"도 결론입니다.
- **함정**: 비율만 보면 **50건짜리 범주와 10만 건짜리 범주가 똑같은 크기의 칸**으로 보입니다 — 건수를 함께 보지 않으면 "탈퇴 고객은 96%가 온라인" 같은 문장을 근거 없이 쓰게 됩니다. `fmt='d'` 는 비율표에서 에러가 납니다(`'.2f'` 로).

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 비율만 보면 표본 크기를 놓친다 -> 건수표와 비율표를 함께 본다
print('[멤버십 x 채널] 건수')
display(pd.crosstab(df['club_member_status'], df['sales_channel_id']))
print('[멤버십 x 채널] 행 기준 비율')
display(pd.crosstab(df['club_member_status'], df['sales_channel_id'], normalize='index').round(3))

ct_news = pd.crosstab(df['fashion_news_frequency'], df['sales_channel_id'], normalize='index')
plt.figure(figsize=(5, 4))
sns.heatmap(ct_news, annot=True, fmt='.2f', cmap='Blues')
plt.title('뉴스 구독 빈도 x 채널 비중')
plt.show()
display(df['fashion_news_frequency'].value_counts())

> 🔧 **강사 Note**: 실제 값 — 멤버십: ACTIVE 온라인 68.6%(108,001건), PRE-CREATE 99.9%(2,199건), LEFT CLUB 96.2%지만 **53건뿐**입니다. 뉴스 구독: NONE 온라인 70.0%(64,019건)와 Regularly 68.3%(49,166건)는 **거의 차이가 없고**, Monthly 만 오프라인 69.7%로 튀는데 **33건뿐**입니다. 비율표만 보면 33건짜리 Monthly 칸이 6만 건짜리 NONE 칸과 같은 크기·같은 진하기로 보여 "구독자는 오프라인을 쓴다"는 결론을 쓰게 됩니다 — 건수표를 함께 봐야 하는 이유가 여기서 드러납니다. 결론으로 쓸 수 있는 것은 "뉴스 구독 여부로는 채널 선호가 갈리지 않는다" 정도이고, 이 표본 자체가 온라인 기록에 치우쳐 있다는 발제문 경고도 함께 적어야 합니다.

### 선택 미션 3 (★★) — 로그 변환·정규화(z-score·Min-Max)로 분포 비교
> 발제문 심화 분석 목표 4

**비즈니스 질문**: 치우친 가격 분포를 **어떻게 손봐야** 고객·상품군끼리 공정하게 비교할 수 있나?

- **무엇을 만드나**: price 에 1) 로그 변환 2) z-score 표준화 3) Min-Max 정규화를 각각 적용해, **네 가지 버전(원본 포함)의 왜도와 히스토그램**을 비교합니다. 그다음 z-score 를 써서 **상품 대분류별 평균 z** 를 구해 어느 군이 전체 평균보다 비싼지 봅니다.
- **어떤 도구를 쓰나**:
  - 로그 변환: `np.log(<값>)`.
  - z-score: 평균을 빼고 표준편차로 나눕니다(`.mean()`·`.std(ddof=1)`).
  - Min-Max: 최솟값을 빼고 (최대−최소)로 나눕니다.
  - 왜도는 `stats.skew(<값>)`, 그림은 2-3 절의 `subplots` + `histplot`.
  - 군별 비교: 새로 만든 z 값을 컬럼으로 붙여(`df.assign(<이름>=…)`) `groupby(<군>)` 으로 평균.
- **이렇게 나오면 맞다**: **로그 변환만 왜도가 크게 줄고, z-score·Min-Max 는 왜도가 원본과 똑같이 남습니다.** 왜 그런지 한 줄로 설명할 수 있으면 이 미션의 핵심을 얻은 것입니다.
- **함정**: z-score·Min-Max 는 축의 **위치와 크기만 바꾸는 선형 변환**이라 분포 **모양**(치우침)은 그대로입니다 — "정규화하면 정규분포가 된다"는 오해를 여기서 깨야 합니다. `np.log` 는 0 이하에서 무한대·NaN 이 됩니다.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 같은 price 를 네 가지로 바꿔 왜도를 비교한다
price_log = np.log(df['price'])
price_z = (df['price'] - df['price'].mean()) / df['price'].std(ddof=1)
price_mm = (df['price'] - df['price'].min()) / (df['price'].max() - df['price'].min())
for name, values in [('원본', df['price']), ('로그', price_log),
                     ('z-score', price_z), ('Min-Max', price_mm)]:
    print(f'{name:8s} 왜도 {stats.skew(values):6.2f}   범위 {values.min():.3f} ~ {values.max():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
sns.histplot(df_s['price'], bins=30, ax=axes[0])
axes[0].set_title('원본')
sns.histplot(np.log(df_s['price']), bins=30, ax=axes[1])
axes[1].set_title('로그 변환')
sns.histplot((df_s['price'] - df['price'].mean()) / df['price'].std(ddof=1), bins=30, ax=axes[2])
axes[2].set_title('z-score 표준화')
plt.tight_layout()
plt.show()

# z 는 '전체 평균에서 몇 표준편차 떨어졌나' 라서 군끼리 비교하기 좋다
z_by_group = (df.assign(price_z=price_z)
                .groupby('index_group_name', observed=True)['price_z']
                .agg(건수='count', 평균z='mean')
                .sort_values('평균z'))
display(z_by_group.round(3))

> 🔧 **강사 Note**: 실제 값 — 왜도: 원본 2.91 / 로그 −0.40 / z-score 2.91 / Min-Max 2.91. **선형 변환(z·Min-Max)은 모양을 못 바꾼다**는 것이 숫자로 드러납니다. 상품 대분류별 평균 z: Baby/Children −0.466, Divided −0.084, Menswear 0.001, Sport 0.033, Ladieswear 0.054 → 아동 상품군이 전체 평균보다 뚜렷하게 저가대입니다. price 는 정규화된 상대값이므로 "몇 크로나 싸다"가 아니라 "평균보다 0.47 표준편차 낮다"로 말합니다.

### 선택 미션 4 (★★) — 이상치 고객·거래 탐색 전략
> 발제문 심화 분석 목표 6

**비즈니스 질문**: "비정상적으로 큰" 거래와 고객을 어떻게 찾고, 찾은 다음 **제거할지 남길지** 어떻게 정하나?

- **무엇을 만드나**: IQR 1.5배 규칙으로 1) **거래 단위** 이상치와 2) **고객 총구매액 단위** 이상치를 각각 찾아, 건수·비율·**그들이 차지하는 매출 비중**을 출력합니다. 그리고 그 이상치를 **제거할지·남길지·따로 표시(플래그)할지**를 근거와 함께 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - IQR 규칙: `.quantile([0.25, 0.75])` 로 Q1·Q3 → IQR → 상한은 Q3 + 1.5 × IQR.
  - 이상치만 고르기: 불리언 조건으로 필터 — 비율은 조건의 `.mean()`, 건수는 `.sum()`.
  - 고객 단위: `df.groupby('customer_id')['price'].sum()` 에 같은 규칙을 적용합니다.
  - 매출 비중: 이상치의 합 ÷ 전체 합.
  - 그림: `sns.boxplot(x=<시리즈>)` — 상자 밖 점이 이 규칙이 말하는 이상치입니다.
  - 어떤 상품군에서 나오는지 보려면 `value_counts().head()`.
- **이렇게 나오면 맞다**: 이상치 거래는 전체의 **5% 안팎인데 매출은 그보다 훨씬 큰 몫**을 차지합니다 — 그래서 "이상치니까 지운다"가 위험하다는 결론이 데이터에서 나옵니다. 거래 단위와 고객 단위의 결과가 다르다는 것도 확인하세요.
- **함정**: **이상치 = 오류가 아닙니다.** 여기서 잡힌 것은 대부분 정상적인 고가 상품(코트·신발)입니다 — 지우면 매출의 상당 부분이 사라집니다. 그리고 IQR 규칙은 **한쪽으로 치우친 분포에서 상단을 과하게 잡는다**는 한계가 있으니, 로그 변환 후 다시 보거나 상위 1% 같은 다른 기준과 비교해 보세요.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# (1) 거래 단위 이상치 — IQR 1.5배 규칙
q1, q3 = df['price'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
out_tx = df[df['price'] > upper]
print(f'Q1 {q1:.4f} / Q3 {q3:.4f} / IQR {iqr:.4f} / 상한 {upper:.4f}')
print(f'이상치 거래 {len(out_tx):,}건 = 전체의 {len(out_tx) / len(df):.2%}, '
      f"매출 점유 {out_tx['price'].sum() / df['price'].sum():.1%}")
display(out_tx['product_group_name'].value_counts().head(5))

# (2) 고객 단위 이상치 — 총구매액에 같은 규칙
cust_total = df.groupby('customer_id')['price'].sum()
cq1, cq3 = cust_total.quantile([0.25, 0.75])
cust_upper = cq3 + 1.5 * (cq3 - cq1)
heavy = cust_total[cust_total > cust_upper]
print(f'고객 총구매액 상한 {cust_upper:.4f}')
print(f'이상치 고객 {len(heavy):,}명 = 전체 고객의 {len(heavy) / len(cust_total):.2%}, '
      f'매출 점유 {heavy.sum() / cust_total.sum():.1%}')

plt.figure(figsize=(8, 3))
sns.boxplot(x=df_s['price'])
plt.title('price 상자그림 — 오른쪽 점들이 IQR 규칙이 말하는 이상치')
plt.show()

> 🔧 **강사 Note**: 실제 값 — 거래: 상한 0.0618, 이상치 5,363건(4.74%)이 **매출의 14.6%**. 상위 제품군은 Garment Upper body 2,487건·Garment Lower body 1,297건 — 오류가 아니라 정상 고가 상품입니다. 고객: 상한 0.0805, 이상치 고객 5,431명(5.84%)이 매출의 19.7%. **전략 예시**: 삭제하지 않고 `is_outlier` 플래그를 달아, 평균을 볼 때는 중앙값·절사평균을 함께 보고 매출 합계에는 포함시킨다.

### 선택 미션 5 (★★) — 월별·분기별 평균 매출 — 추세와 시즌성
> 발제문 심화 분석 목표 5

**비즈니스 질문**: 매출이 특정 시기에 몰리는가? 그 급증을 "성장"이라고 불러도 되는가?

- **무엇을 만드나**: 월별·분기별로 **총매출·건수·평균가**를 집계해 표로 만들고, 월별 흐름을 꺾은선으로, 분기별 비교를 막대로 그립니다. 그리고 **급증한 달이 건수 때문인지 단가 때문인지** 구분해 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 분기 파생: 날짜형 컬럼의 `.dt.quarter` (1~4).
  - 집계: `groupby(<월 또는 분기>).agg(…)` 로 합계·건수·평균을 한 표에.
  - 그림: 2-3 절의 `lineplot`(추이) + `barplot`(분기 비교). 집계 결과는 `reset_index()` 후 넣습니다.
  - 최대·최소 달 찾기: `.idxmax()`·`.idxmin()`.
- **이렇게 나오면 맞다**: 총매출이 가장 큰 달과 평균가가 가장 큰 달이 **서로 다릅니다** — 그래서 "매출이 늘었다"를 **건수 증가**와 **단가 상승**으로 쪼개 설명할 수 있으면 성공입니다.
- **함정**: 이 데이터는 **1년뿐**이라 "증가 추세/감소 추세" 같은 장기 판단을 할 수 없습니다(작년 같은 달과 비교할 수 없으므로). 시즌 이벤트(블랙프라이데이·연말연시)로 인한 급증을 성장으로 단정하지 말라는 발제문 경고가 바로 이 지점입니다.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
df['quarter'] = df['t_dat'].dt.quarter
monthly = df.groupby('month').agg(
    건수=('price', 'count'), 총매출=('price', 'sum'), 평균가=('price', 'mean'))
quarterly = df.groupby('quarter').agg(
    건수=('price', 'count'), 총매출=('price', 'sum'), 평균가=('price', 'mean'))
display(monthly.round(4))
display(quarterly.round(4))
print('총매출 최대 달:', monthly['총매출'].idxmax(), '/ 평균가 최대 달:', monthly['평균가'].idxmax())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=monthly.reset_index(), x='month', y='총매출', marker='o', ax=axes[0])
axes[0].set_title('월별 총매출')
axes[0].set_xticks(range(1, 13))
sns.barplot(data=monthly.reset_index(), x='month', y='평균가', ax=axes[1])
axes[1].set_title('월별 평균가(거래 단가)')
plt.tight_layout()
plt.show()

> 🔧 **강사 Note**: 실제 값 — 총매출 최대는 6월(340.0, 건수 13,286건), 최소는 8월(217.7). 평균가는 11월·9월이 0.0325 로 가장 높고 7월이 0.0222 로 가장 낮습니다. 분기 총매출은 Q2 935.0 > Q3 769.4 > Q4 733.8 > Q1 699.4. **읽기**: 6~7월은 **건수가 많고 단가는 낮아** 여름 세일 성격이고, 11월은 **건수는 보통인데 단가가 높아** 연말 고가 상품(외투) 구매로 보입니다. 1년 데이터라 "Q2 가 성수기"라고 일반화할 수 없습니다.

### 선택 미션 6 (★★) — 채널 비중 해석 보정 — 비율(%)과 ARPU
> 발제문 심화 분석 목표 FAQ(표본 크기 차이 보정)

**비즈니스 질문**: 온라인이 오프라인보다 매출이 크다는 말은, 온라인 **기록이 더 많아서** 생긴 착시가 아닐까?

- **무엇을 만드나**: 채널별로 **건수·총매출·고객 수**를 구하고, 여기서 1) 건수 비중(%) 2) 매출 비중(%) 3) **ARPU(고객당 평균 구매액)** 4) 고객당 구매 건수 5) 거래당 평균가를 계산한 표를 만듭니다. 그리고 **표본 크기 차이를 보정한 뒤에도 결론이 유지되는지** 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 고객 수는 중복을 뺀 개수 — `agg(<이름>=('customer_id', 'nunique'))`.
  - 비중: 각 값을 `<컬럼>.sum()` 으로 나눕니다.
  - ARPU = 총매출 ÷ 고객 수, 고객당 건수 = 건수 ÷ 고객 수, 거래당 평균가 = 총매출 ÷ 건수.
  - 두 채널을 모두 쓴 고객 수: `df.groupby('customer_id')['sales_channel_id'].nunique()` 가 2 인 고객을 셉니다.
- **이렇게 나오면 맞다**: 채널별 ARPU 와 거래당 평균가를 나란히 놓았을 때, **"온라인 우세"가 단순히 기록 수 차이 때문이 아니라는 것(또는 그 반대)** 을 근거를 들어 말할 수 있으면 성공입니다.
- **함정**: **채널별 고객 수를 더하면 전체 고객 수보다 많습니다** — 두 채널을 모두 쓴 고객이 양쪽에 세어지기 때문입니다(그래서 ARPU 는 채널별로만 비교하고 합산하지 마세요). 그리고 이 데이터는 애초에 온라인 기록이 더 많이 수집된 표본이라, 보정 후에도 **"시장 전체에서 온라인이 우세하다"로 일반화할 수는 없습니다.**

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
ch = df.groupby('sales_channel_id').agg(
    건수=('price', 'count'), 총매출=('price', 'sum'), 고객수=('customer_id', 'nunique'))
ch['건수비중'] = (ch['건수'] / ch['건수'].sum()).round(3)
ch['매출비중'] = (ch['총매출'] / ch['총매출'].sum()).round(3)
ch['ARPU'] = (ch['총매출'] / ch['고객수']).round(4)          # 고객당 평균 구매액
ch['고객당_건수'] = (ch['건수'] / ch['고객수']).round(3)
ch['거래당_평균가'] = (ch['총매출'] / ch['건수']).round(4)
display(ch)

ch_per_cust = df.groupby('customer_id')['sales_channel_id'].nunique()
print(f'두 채널을 모두 쓴 고객: {(ch_per_cust == 2).sum():,}명 '
      f'/ 전체 고객 {df["customer_id"].nunique():,}명')
print('-> 채널별 고객수를 더하면 전체 고객수보다 많다(중복). ARPU 는 채널 안에서만 비교한다.')

> 🔧 **강사 Note**: 실제 값 — 오프라인(1): 34,823건 / 매출 793.85 / 고객 32,031명 / ARPU 0.0248 / 고객당 1.087건 / 거래당 0.0228. 온라인(2): 78,395건 / 매출 2,343.75 / 고객 64,291명 / ARPU 0.0365 / 고객당 1.219건 / 거래당 0.0299. 건수 비중 30.8:69.2, 매출 비중 25.3:74.7. 두 채널 모두 쓴 고객 3,269명. **읽기**: 건수를 빼고 고객 1명 기준으로 보정해도 온라인 ARPU 가 1.47배 높습니다 — 즉 기록 수 차이만으로는 설명되지 않습니다. 다만 표본 자체가 온라인 편향이라 시장 전체로 일반화하면 안 됩니다.

### 선택 미션 7 (★★★) — 고객별 총 구매액 상위 20% vs 하위 20%
> 발제문 심화 분석 목표 2

**비즈니스 질문**: 우리 매출은 소수의 고객에게 얼마나 몰려 있고, 그 고객들은 무엇이 다른가?

- **무엇을 만드나**: 고객별 **총구매액·구매건수·평균구매액·온라인 비중**을 만든 뒤 총구매액으로 **5분위**로 나눠, **상위 20%와 하위 20%** 의 지표를 비교하는 표와 그림을 만듭니다. 두 그룹이 **전체 매출에서 차지하는 비중**도 함께 계산합니다.
- **어떤 도구를 쓰나**:
  - 고객 단위 표: `df.groupby('customer_id').agg(…)` — 온라인 비중은 "채널이 2인가"라는 **참/거짓 컬럼의 평균**으로 구할 수 있습니다(`df.assign(<이름>=<조건>)` 으로 먼저 컬럼을 만들면 편합니다).
  - 5분위 나누기: `pd.qcut(<시리즈>, q=5, labels=[…])` — 같은 **개수**로 나눕니다(`pd.cut` 은 같은 **폭**으로 나눔).
  - 그룹 비교: `groupby(<그룹 컬럼>, observed=True)[<지표들>].mean()`.
  - 매출 점유율: 그룹별 합계 ÷ 전체 합계.
  - 그림: 2-3 절의 `barplot` 으로 그룹별 지표 비교.
- **이렇게 나오면 맞다**: 상위 20% 가 전체 매출의 **40% 이상**을 차지하고, 하위 20% 는 10% 미만입니다. 그리고 상위 그룹은 평균구매액·구매건수·온라인 비중이 **모두** 더 높게 나옵니다 — 세 지표 중 어느 것이 가장 크게 벌어지는지 말할 수 있으면 성공입니다.
- **함정**: `qcut` 은 **같은 개수**로 나누므로, 총구매액이 같은 고객이 많으면 경계가 딱 20%가 아닐 수 있습니다. 그리고 이 5분위는 **총구매액으로 만든 것**이라 "상위 20%가 총구매액이 크다"는 것은 동어반복입니다 — 의미 있는 발견은 **평균구매액·건수·채널처럼 나누는 데 쓰지 않은 지표**에서 나옵니다.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 고객 단위 지표 — 온라인 비중은 '채널==2' 참/거짓 컬럼의 평균
cust = (df.assign(온라인=(df['sales_channel_id'] == 2))
          .groupby('customer_id')
          .agg(총구매액=('price', 'sum'), 구매건수=('price', 'count'),
               온라인비중=('온라인', 'mean')))
cust['평균구매액'] = cust['총구매액'] / cust['구매건수']

cust['그룹'] = pd.qcut(cust['총구매액'], q=5,
                     labels=['하위20%', '2분위', '3분위', '4분위', '상위20%'])
cmp_tbl = cust.groupby('그룹', observed=True)[
    ['총구매액', '평균구매액', '구매건수', '온라인비중']].mean()
display(cmp_tbl.round(4))

share = cust.groupby('그룹', observed=True)['총구매액'].sum() / cust['총구매액'].sum()
display(share.round(3))

plt.figure(figsize=(11, 3.6))
for i, col in enumerate(['평균구매액', '구매건수', '온라인비중'], start=1):
    plt.subplot(1, 3, i)
    sns.barplot(x=cmp_tbl.index, y=cmp_tbl[col])
    plt.title(col)
    plt.xticks(rotation=30, ha='right')
    plt.xlabel('')
plt.tight_layout()
plt.show()

> 🔧 **강사 Note**: 실제 값 — 상위20%: 총구매액 0.0751 / 평균구매액 0.0500 / 구매건수 1.74 / 온라인 비중 0.810. 하위20%: 0.0104 / 0.0103 / 1.01 / 0.533. 배수로는 총액 7.2배, **단가 4.84배**, 건수 1.7배 — 즉 이 데이터에서 상·하위 격차는 "많이 사서"보다 **"비싼 것을 사서"** 생깁니다. 매출 점유율은 상위20% 44.5% / 하위20% 6.7%. 온라인 비중이 0.81 대 0.53 으로 벌어지는 것도 눈에 띕니다(단, 인과가 아니라 관찰된 연관입니다).

### 선택 미션 8 (★★★) — 월별 매출의 3개월 롤링 평균
> 발제문 심화 분석 목표 3

**비즈니스 질문**: 월별 매출이 들쭉날쭉할 때, **노이즈를 걷어낸 흐름**은 어떤 모양인가?

- **무엇을 만드나**: 월별 총매출 표를 만들고 **3개월 롤링 평균** 컬럼을 추가해, 원래 선과 롤링 선을 **한 그림에 겹쳐** 그립니다. 앞의 두 달이 왜 비어 있는지, 롤링 창을 6개월로 늘리면 무엇이 달라지는지도 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 월별 집계: `df.groupby('month')['price'].sum()` — 그래프에 넣으려면 `reset_index(name=<이름>)`.
  - 롤링 평균: `<시리즈>.rolling(<창 크기>).mean()` — 창 크기 3 이면 3개월 이동 평균.
  - 겹쳐 그리기: `plt.plot(<x>, <y>, label=…)` 을 두 번 부르고 `plt.legend()`.
  - 월 눈금 고정: `plt.xticks(range(1, 13))`.
- **이렇게 나오면 맞다**: 롤링 선이 원래 선보다 **매끄럽고**, 1·2월 자리는 값이 없어 선이 3월부터 시작합니다. 두 선이 어긋나는 달을 짚어 "그 달만 튀었다"고 말할 수 있으면 성공입니다.
- **함정**: 롤링 평균은 **창 크기−1 개월만큼 뒤로 늦게 반응**합니다(그래서 앞부분이 NaN). 우리 데이터는 12개월뿐이라 6개월 창을 쓰면 점이 7개만 남아 추세라고 부르기 어렵습니다 — **3개월 창이 이 데이터에 맞는 선택**입니다. 그리고 월별 합계는 **그 달의 일수·표본 수**에 영향을 받으니, 필요하면 평균가와 함께 보세요.

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
monthly_sales = df.groupby('month')['price'].sum().reset_index(name='총매출')
monthly_sales['롤링3'] = monthly_sales['총매출'].rolling(3).mean()
display(monthly_sales.round(2))

plt.figure(figsize=(9, 4))
plt.plot(monthly_sales['month'], monthly_sales['총매출'],
         alpha=0.45, marker='o', label='월별 총매출')
plt.plot(monthly_sales['month'], monthly_sales['롤링3'],
         color='crimson', marker='o', label='3개월 롤링 평균')
plt.xticks(range(1, 13))
plt.title('월별 총매출과 3개월 롤링 평균')
plt.xlabel('월')
plt.ylabel('총매출(정규화된 상대값의 합)')
plt.legend()
plt.show()
print('롤링 최대:', monthly_sales.loc[monthly_sales['롤링3'].idxmax(), 'month'], '월')

> 🔧 **강사 Note**: 실제 값 — 월별 총매출 229.8(1월) … 340.0(6월) … 218.8(12월), 3개월 롤링은 3월 233.13에서 시작해 **6월 311.68로 최대**, 이후 12월 244.59까지 내려옵니다. 원래 선의 6월 급등(340.0)이 롤링에서는 311.68로 눌리는 것이 "노이즈 제거"의 효과입니다. 1~2월은 앞 데이터가 없어 NaN — 이 결측을 "매출 0"으로 오해하지 않게 그래프에서 확인하세요.

## 5. 관찰 정리 (서술)
**이 절에서 할 일**: 위에서 본 집계·그래프를 근거로 **한 문단짜리 인사이트**를 씁니다. "A 이다" 로 끝내지 말고, **다른 설명 가능성**(다른 변수가 진짜 원인일 수도 있다는 점)도 함께 짚으세요 — 관찰연구는 상관이지 인과가 아닙니다.

**쓰는 순서**: 1) 어떤 집계·그래프를 봤나 → 2) 거기서 무엇을 관찰했나 → 3) 다른 설명 가능성은 없는가 → 4) 우연인지 검정으로 확인하고 싶은 질문.

- 어떤 축(연령대·채널·상품군 등)에서 가장 뚜렷한 차이를 봤나요?
- 그 차이가 우연이 아니라고 자신 있게 말할 수 있나요, 아니면 검정으로 확인하고 싶은가요?
- 표본 크기가 작은 범주나, 온라인에 치우친 이 데이터의 한계를 함께 적었나요?

**예시 답안 — 이렇게 쓸 수 있다는 한 사례입니다**

연령대별 집계(3절 미션 2)를 보면 **20대의 거래 건수와 총매출이 압도적으로 크고**, 평균가는 연령대가 높아질수록 완만하게 오른다. 채널 교차표에서는 **모든 연령대에서 온라인(채널 2) 비중이 오프라인보다 높고**, 특히 20대에서 그 격차가 가장 크다. 고객을 총구매액 5분위로 나눠 보면(선택 미션 7) **상위 20%가 전체 매출의 44.5%** 를 차지하는데, 그 격차는 구매 건수(1.7배)보다 **거래 단가(4.84배)** 에서 나온다. 다만 이 둘을 곧바로 "나이가 채널 선택에 영향을 준다"고 해석하긴 이르다 — **20대는 가입 시점 자체가 최근이라 애초에 온라인 중심 마케팅에 더 많이 노출됐을 수도 있고**, 멤버십 상태나 거주 지역 같은 다른 변수가 함께 얽혀 있을 수 있다. 채널 비중도 **이 표본이 온라인 기록을 더 많이 담고 있다는 한계**를 감안해야 하며, 고객 1명 기준으로 보정한 ARPU(0.0365 대 0.0248)까지 확인한 뒤에야 "기록 수 차이만은 아니다"라고 말할 수 있다. 이 차이들이 "충분히 큰 표본에서도 우연이 아닌지"는 아직 확인하지 않았다 — **연령대별 평균가 차이가 통계적으로 유의한지(ANOVA/Kruskal-Wallis)**, **연령대와 채널 선호가 독립적인지(카이제곱)** 를 다음 파트(03_데이터_통계)에서 검정으로 확인하고 싶다.

> ✅ **여기까지 되면 통과**: 한 문단 서술에 관찰 + 다른 설명 가능성 + 다음 파트로 넘길 질문이 모두 있으면 됩니다.